# CHE1147 Group 6 — Notebook 03: Results and Interpretation

**Research question:** *To what extent are the most influential molecular descriptors shared across the five Tox21 stress-response assays?*

This notebook implements **Section 3 — Results and Interpretation** as a single continuous section with numbered subsections. It continues from Notebook 02 and **does not retrain any model**.

**Feature importance (Sections 3.2–3.6):**
- **Logistic Regression (LR):** rank SFS-selected descriptors by **absolute standardized coefficient**, retaining coefficient signs for interpretation.
- **Random Forest (RF):** rank descriptors by **mean decrease in Average Precision (AP)** after permutation on the held-out scaffold-disjoint test set.
- Save the **top 10** descriptors for each model and assay.

**Pairwise Jaccard similarity (Sections 3.7–3.10):**
- **Primary:** compare RF top-10 descriptor identities across the five assays (10 unique assay pairs).
- **Secondary:** repeat the cross-assay Jaccard comparison for LR.
- **Supporting:** compare LR top 10 versus RF top 10 *within* each assay.

**Shared-descriptor analysis (Sections 3.11–3.14):**
- Count each descriptor's top-10 appearances across the **five assays**, separately for RF and LR.
- Report counts and percentages of unique descriptors present in **exactly** or **at least** 1, 2, 3, 4 or 5 assays.
- Generate descriptor-by-assay presence heatmaps and frequency-distribution figures.

**Inputs:** `data/processed/tox21_clean.csv` and the saved `.joblib` models, predictions, selected-feature files, split assignments, and performance summary from Notebook 02.

**Outputs:** CSV tables under `results/metrics/Results_Interpretation/` and PNG figures under `results/figures/Results_Interpretation/`.

**Important:** Interpret these as *descriptive, model-dependent* overlap statistics. Jaccard is not a hypothesis test, and shared descriptors do not establish shared causal biology. The held-out test set is used for **post-hoc** RF interpretation only, never to retune models.

## 3.0 — Imports, paths, and settings

Place this notebook in your repository's `notebooks/` directory. It locates the repository by searching for the Notebook 01 cleaned dataset. To match saved model serialization, use the **same scikit-learn environment as Notebook 02** (the uploaded results report scikit-learn 1.7.2).

`N_PERM_REPEATS=10` is intended for the final feature-importance analysis. You may temporarily set it to 2 while testing the pipeline, then rerun with 10 for final outputs.

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import joblib
from joblib import parallel_backend
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

ASSAYS = ["SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"]
MODELS = ["LR", "RF"]
TOP_N = 10
N_PERM_REPEATS = 10
PERM_JOBS = 4                 # feature-level threads, no additional forest parallelism
RANDOM_SEED = 42
FIGURE_DPI = 300

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "processed" / "tox21_clean.csv").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError(
        "Missing data/processed/tox21_clean.csv. "
        "Run Notebook 01, then open this notebook within your CHE1147 repository."
    )

input_path = root / "data" / "processed" / "tox21_clean.csv"
metrics02 = root / "results" / "metrics" / "Preprocessing_Models_Evaluation"
models02 = root / "results" / "models" / "Preprocessing_Models_Evaluation"
metrics03 = root / "results" / "metrics" / "Results_Interpretation"
figures03 = root / "results" / "figures" / "Results_Interpretation"
metrics03.mkdir(parents=True, exist_ok=True)
figures03.mkdir(parents=True, exist_ok=True)

for required in [metrics02 / "split_assignments.csv",
                 metrics02 / "model_performance.csv",
                 metrics02 / "run_manifest.json"]:
    if not required.is_file():
        raise FileNotFoundError(f"Missing Notebook 02 output: {required}")

manifest = json.loads((metrics02 / "run_manifest.json").read_text(encoding="utf-8"))
if manifest.get("run_mode") != "full":
    raise ValueError("Notebook 02 outputs must come from RUN_MODE='full'.")
if set(ASSAYS) != set(manifest.get("assays", [])):
    raise ValueError("The manifest does not cover all five planned assays.")
if manifest.get("sfs_min_improvement") != 0.001:
    print("Note: manifest SFS threshold differs from expected 0.001:",
          manifest.get("sfs_min_improvement"))
if manifest.get("scikit_learn_version") != sklearn.__version__:
    print("WARNING: Notebook 02 used scikit-learn",
          manifest.get("scikit_learn_version"),
          "but Notebook 03 has", sklearn.__version__,
          "— model persistence across versions is not guaranteed.")

print("Repository root:", root)
print("Notebook 02 models:", models02)
print("Notebook 03 metrics:", metrics03)
print("Notebook 03 figures:", figures03)

## 3.1 — Reconstruct the *exact* held-out test set and verify saved predictions

Notebook 02's `split_assignments.csv` records each molecule's original row index (`source_row`), assay, activity label, and scaffold-based train/test assignment. We recover the test data using those **saved row indices**, **not** a newly computed random split.

We also verify that each portable saved pipeline reproduces Notebook 02's predicted probabilities and test AP. Any mismatch raises an error so we do not silently interpret the wrong model or data order.

In [ ]:
clean = pd.read_csv(input_path, low_memory=False).reset_index(drop=True)
splits = pd.read_csv(metrics02 / "split_assignments.csv")
performance = pd.read_csv(metrics02 / "model_performance.csv")

required_split_cols = {"assay", "split", "source_row", "canonical_smiles", "label", "scaffold_group"}
if not required_split_cols.issubset(splits.columns):
    raise ValueError(f"Missing split columns: {required_split_cols - set(splits.columns)}")
if not clean["canonical_smiles"].is_unique or clean["canonical_smiles"].isna().any():
    raise ValueError("Notebook 01 cleaned data must contain unique nonmissing canonical SMILES.")

# Keep the exact original CSV row order, as required by Notebook 02's source_row.

def load_assay_test(assay):
    split = splits.loc[(splits["assay"] == assay) &
                       (splits["split"] == "test")].copy().reset_index(drop=True)
    if split.empty:
        raise ValueError(f"No held-out test molecules found for {assay}")

    row_ids = split["source_row"].to_numpy(dtype=int)
    if len(np.unique(row_ids)) != len(row_ids):
        raise ValueError(f"Duplicate test source_row in {assay}")
    if np.any(row_ids < 0) or np.any(row_ids >= len(clean)):
        raise ValueError(f"Invalid source_row index for {assay}")

    test = clean.iloc[row_ids].copy().reset_index(drop=True)
    if not np.array_equal(test["canonical_smiles"].astype(str).to_numpy(),
                          split["canonical_smiles"].astype(str).to_numpy()):
        raise ValueError(f"Clean dataset row order has changed for {assay}")
    y_test = split["label"].to_numpy(dtype=int)
    if not np.array_equal(pd.to_numeric(test[assay]).to_numpy(dtype=int), y_test):
        raise ValueError(f"Assay labels do not match split_assignments for {assay}")
    if len(np.unique(y_test)) != 2:
        raise ValueError(f"Need both classes on test set for AP: {assay}")
    return test, y_test, split


def get_model_features(pipeline):
    """Get ordered input column names from the fitted portable ColumnTransformer."""
    if not all(k in pipeline.named_steps for k in ["preprocess", "model"]):
        raise ValueError("Expected portable pipeline steps 'preprocess' and 'model'.")
    transformers = pipeline.named_steps["preprocess"].transformers_
    if len(transformers) < 1 or transformers[0][0] != "selected_descriptors":
        raise ValueError("Unexpected Notebook 02 ColumnTransformer format.")
    return list(transformers[0][2])


def load_checked_model(assay, model_name, test, y_test, split):
    path = models02 / f"{model_name}_{assay}.joblib"
    features_path = metrics02 / f"selected_features_{model_name}_{assay}.csv"
    pred_path = metrics02 / f"predictions_{assay}_{model_name}.csv"
    for file in [path, features_path, pred_path]:
        if not file.is_file():
            raise FileNotFoundError(file)

    pipeline = joblib.load(path)  # Only load your own trusted Notebook 02 files.
    feature_names = get_model_features(pipeline)
    saved_features = pd.read_csv(features_path)["descriptor"].tolist()
    if feature_names != saved_features:
        raise ValueError(f"Saved {model_name} descriptor order mismatch for {assay}.")
    missing = sorted(set(feature_names) - set(test.columns))
    if missing:
        raise ValueError(f"Missing required {model_name} descriptors: {missing[:10]}")

    # Mirrors Notebook 02's cleaning of nonfinite descriptor values.
    X_test = test.loc[:, feature_names].apply(pd.to_numeric, errors="coerce")
    X_test = X_test.replace([np.inf, -np.inf], np.nan)

    predicted = pipeline.predict_proba(X_test)[:, 1]
    saved = pd.read_csv(pred_path)
    if not np.array_equal(saved["canonical_smiles"].astype(str).to_numpy(),
                          split["canonical_smiles"].astype(str).to_numpy()):
        raise ValueError(f"Prediction order differs from test split: {assay} {model_name}")
    if not np.array_equal(saved["y_true"].to_numpy(dtype=int), y_test):
        raise ValueError(f"Prediction labels differ from test split: {assay} {model_name}")
    np.testing.assert_allclose(
        predicted, saved["predicted_probability"].to_numpy(dtype=float),
        atol=1e-7, rtol=1e-6,
        err_msg=f"Saved probabilities do not match fitted {model_name} model for {assay}",
    )
    ap = float(average_precision_score(y_test, predicted))
    recorded = performance.loc[(performance["assay"] == assay) &
                               (performance["model"] == model_name)]
    if len(recorded) != 1:
        raise ValueError(f"Expected exactly one performance row for {assay} {model_name}")
    np.testing.assert_allclose(ap, float(recorded["ap"].iloc[0]), atol=1e-7, rtol=1e-6)
    return pipeline, X_test, feature_names, ap

# Cache the validated objects for the subsequent cells.
assay_test = {}
models = {}
validation_rows = []
for assay in ASSAYS:
    test, y_test, split = load_assay_test(assay)
    assay_test[assay] = {"test": test, "y": y_test, "split": split}
    for model_name in MODELS:
        fitted, X, columns, ap = load_checked_model(assay, model_name, test, y_test, split)
        models[(assay, model_name)] = {
            "pipeline": fitted, "X_test": X, "features": columns, "test_ap": ap
        }
        validation_rows.append({
            "assay": assay, "model": model_name, "n_test": len(y_test),
            "n_features": len(columns), "test_ap_verified": ap,
        })

print("Verified: all 10 saved LR/RF pipelines reproduce Notebook 02 test predictions and AP.")
display(pd.DataFrame(validation_rows).round(5))

## 3.2 — Logistic Regression: coefficients of standardized descriptors

Notebook 02's portable LR pipelines contain `ColumnTransformer → SimpleImputer → StandardScaler → LogisticRegression`. Therefore `model.coef_[0]` already corresponds to **one standardized unit** in each descriptor. **Do not standardize the coefficients again.**

- Rank by `abs(coefficient)` (largest first), **not** SFS selection order.
- Preserve signed coefficient and `exp(coefficient)` to interpret association with the odds of assay activity, holding other selected descriptors constant.
- The coefficient is a *model association*, not a causal claim.

In [ ]:
lr_importance_tables = {}

for assay in ASSAYS:
    info = models[(assay, "LR")]
    pipeline = info["pipeline"]
    features = info["features"]
    estimator = pipeline.named_steps["model"]
    preprocess = pipeline.named_steps["preprocess"]

    # Confirm the actual fitted preprocessing contains StandardScaler.
    feature_transform = preprocess.named_transformers_["selected_descriptors"]
    if "scale" not in feature_transform.named_steps:
        raise ValueError(f"No StandardScaler in saved LR pipeline for {assay}")
    if not isinstance(feature_transform.named_steps["scale"], StandardScaler):
        raise TypeError(f"Unexpected LR scaler for {assay}")
    if list(estimator.classes_) != [0, 1]:
        raise ValueError(f"Unexpected LR class encoding for {assay}")

    coefficients = np.asarray(estimator.coef_[0], dtype=float)
    if len(coefficients) != len(features):
        raise ValueError(f"LR coefficient/descriptor count mismatch for {assay}")

    sfs_table = pd.read_csv(metrics02 / f"selected_features_LR_{assay}.csv")
    df = pd.DataFrame({
        "assay": assay,
        "model": "LR",
        "descriptor": features,
        "selection_order": sfs_table["selection_order"].to_numpy(dtype=int),
        "coefficient_standardized": coefficients,
        "abs_coefficient": np.abs(coefficients),
        "odds_ratio_per_1sd": np.exp(coefficients),
    })
    df = df.sort_values(
        ["abs_coefficient", "descriptor"], ascending=[False, True], kind="stable"
    ).reset_index(drop=True)
    df.insert(2, "rank", np.arange(1, len(df) + 1))
    df["is_top10"] = df["rank"] <= TOP_N

    out = metrics03 / f"feature_importance_LR_{assay}.csv"
    df.to_csv(out, index=False)
    lr_importance_tables[assay] = df
    print(f"{assay}: {len(df)} LR features; saved {out.name}")
    display(df.head(TOP_N)[[
        "rank", "descriptor", "coefficient_standardized", "abs_coefficient"
    ]].round(4))

## 3.3 — Random Forest: held-out permutation importance using AP

For each saved RF model, `permutation_importance`:

1. Measures the original **test AP** on the existing scaffold-disjoint holdout.
2. Shuffles the values of **one raw RDKit descriptor** between test molecules; all other columns and activity labels remain unchanged.
3. Recalculates AP with the **same fitted model**; repeats this `N_PERM_REPEATS` times.
4. Returns the average AP decrease and its standard deviation, separately for every retained RF descriptor.

Large positive values indicate strong model reliance on a descriptor **on this held-out dataset**; near-zero or negative importance should not be overinterpreted. Correlated descriptors can split or mask importance. **No model fitting takes place here.**

**Runtime:** RF has ~212–213 input features per assay, so this is the computationally expensive part. The code uses 4 feature-parallel threads to avoid serializing the large RF models to several processes.

In [ ]:
rf_importance_tables = {}

for assay_idx, assay in enumerate(ASSAYS):
    info = models[(assay, "RF")]
    fitted = info["pipeline"]
    X_test = info["X_test"]
    y_test = assay_test[assay]["y"]
    features = info["features"]
    original_ap = info["test_ap"]
    start = time.perf_counter()

    print(f"\n{assay} RF: permutation importance on {len(y_test):,} held-out molecules "
          f"× {len(features)} descriptors × {N_PERM_REPEATS} shuffles", flush=True)
    with parallel_backend("threading"):
        result = permutation_importance(
            estimator=fitted,
            X=X_test,
            y=y_test,
            scoring="average_precision",
            n_repeats=N_PERM_REPEATS,
            random_state=RANDOM_SEED + assay_idx,
            n_jobs=PERM_JOBS,
        )

    df = pd.DataFrame({
        "assay": assay,
        "model": "RF",
        "descriptor": features,
        "baseline_test_ap": original_ap,
        "permutation_ap_decrease_mean": result.importances_mean,
        "permutation_ap_decrease_std": result.importances_std,
        "n_repeats": N_PERM_REPEATS,
        "random_seed": RANDOM_SEED + assay_idx,
    })
    df = df.sort_values(
        ["permutation_ap_decrease_mean", "descriptor"],
        ascending=[False, True], kind="stable"
    ).reset_index(drop=True)
    df.insert(2, "rank", np.arange(1, len(df) + 1))
    df["is_top10"] = df["rank"] <= TOP_N

    out = metrics03 / f"feature_importance_RF_{assay}.csv"
    df.to_csv(out, index=False)
    rf_importance_tables[assay] = df
    elapsed = (time.perf_counter() - start) / 60
    print(f"{assay}: saved {out.name}; elapsed {elapsed:.1f} min")
    display(df.head(TOP_N)[[
        "rank", "descriptor", "permutation_ap_decrease_mean",
        "permutation_ap_decrease_std"
    ]].round(4))
    if (df.head(TOP_N)["permutation_ap_decrease_mean"] <= 0).any():
        print("CAUTION: Some top-10 RF descriptors have nonpositive mean AP decrease. "
              "A fixed top-10 list does not establish that all 10 features are useful.")

## 3.4 — Create the common top-10 descriptor table for Jaccard analysis

The top 10 are **model-specific**:
- LR: largest absolute standardized coefficients;
- RF: largest mean permutation AP decreases.

We compare descriptor **identities** across models and assays, **not** the numerical coefficient and permutation-AP scales.

In [ ]:
top_tables = []
for assay in ASSAYS:
    for model_name, table, score_col, method in [
        ("LR", lr_importance_tables[assay], "abs_coefficient", "absolute_standardized_coefficient"),
        ("RF", rf_importance_tables[assay], "permutation_ap_decrease_mean", "mean_permutation_ap_decrease"),
    ]:
        top = table.head(TOP_N).loc[:, ["assay", "model", "rank", "descriptor", score_col]].copy()
        if len(top) != TOP_N or top["descriptor"].duplicated().any():
            raise ValueError(f"Cannot form a valid top-{TOP_N} descriptor set: {assay} {model_name}")
        top = top.rename(columns={score_col: "importance_value"})
        top["importance_method"] = method
        top_tables.append(top)

top_descriptors = pd.concat(top_tables, ignore_index=True)
top_descriptors.to_csv(metrics03 / "top_descriptors_by_assay.csv", index=False)

assert len(top_descriptors) == len(ASSAYS) * len(MODELS) * TOP_N
print("Saved top_descriptors_by_assay.csv (100 rows: 5 assays × 2 models × 10 features).")
display(top_descriptors.head(20))

## 3.5 — Save importance plots (PNG)

The LR plot shows **signed standardized coefficients**, keeping the association direction visible. The RF plot shows **mean AP decrease ± one standard deviation** across shuffles. Plot order uses the **top-10 ranking** determined above.

In [ ]:
for assay in ASSAYS:
    # LR: signed coefficients for the top 10 ranked by absolute magnitude.
    lr_top = lr_importance_tables[assay].head(TOP_N).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8.0, 5.3))
    ax.barh(lr_top["descriptor"], lr_top["coefficient_standardized"])
    ax.axvline(0, linewidth=1, color="black")
    ax.set_title(f"{assay} — LR top {TOP_N} standardized coefficients")
    ax.set_xlabel("Standardized coefficient (change in log-odds per 1 SD)")
    ax.set_ylabel("Molecular descriptor")
    fig.tight_layout()
    fig.savefig(figures03 / f"feature_importance_LR_{assay}.png",
                dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

    # RF: mean test AP decrease after shuffling one descriptor (uncertainty shown).
    rf_top = rf_importance_tables[assay].head(TOP_N).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8.0, 5.3))
    ax.barh(
        rf_top["descriptor"], rf_top["permutation_ap_decrease_mean"],
        xerr=rf_top["permutation_ap_decrease_std"], capsize=3,
    )
    ax.axvline(0, linewidth=1, color="black")
    ax.set_title(f"{assay} — RF top {TOP_N} permutation importances")
    ax.set_xlabel("Mean decrease in test Average Precision (AP)")
    ax.set_ylabel("Molecular descriptor")
    fig.tight_layout()
    fig.savefig(figures03 / f"feature_importance_RF_{assay}.png",
                dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

print("Saved 10 importance plots in", figures03)

## 3.6 — Interpretation guidelines

- LR top-10 rankings use **absolute standardized coefficients**, while retaining the sign for chemical interpretation.
- RF top-10 rankings use **mean permutation AP decrease** measured with the already-finalized model on the held-out test data.
- We compare the **identities** of top-10 descriptors, not coefficient values against AP decreases.
- Correlated RDKit descriptors may have diluted or unstable individual importance; near-ties around rank 10 may affect Jaccard scores.
- Feature importance is *associational and model-dependent*. Neither cross-assay overlap nor coefficient signs establish a biological causal mechanism.

The following subsections use `top_descriptors_by_assay.csv` directly. They do **not** repeat training or permutations.

---

## 3.7 — Pairwise Jaccard similarity: assay overlap and model agreement

For any two top-10 descriptor sets, Jaccard similarity is

\[
J(A,B)=\frac{|A\cap B|}{|A\cup B|}.
\]

Each of the five assays is compared with every other assay (**10 unique pairs**), separately for RF (primary) and LR (secondary). A separate calculation compares LR against RF *within* each assay (supporting). **Only descriptor names** are compared; LR coefficients and RF AP decreases are never compared numerically.

The main outputs are `jaccard_similarity_RF.csv`, `jaccard_similarity_LR.csv`, and `jaccard_similarity_LR_vs_RF.csv`, plus detailed pairwise CSV tables and plots.

In [ ]:
from itertools import combinations

# Reuse the top-10 table produced at the end of Section 3.
# Validate its structure to catch incomplete model runs or accidental duplicates.
required_top_cols = {"assay", "model", "rank", "descriptor"}
if not required_top_cols.issubset(top_descriptors.columns):
    raise ValueError(f"Missing top-descriptor columns: {required_top_cols - set(top_descriptors.columns)}")

expected_groups = {(model, assay) for model in MODELS for assay in ASSAYS}
actual_groups = set(zip(top_descriptors["model"], top_descriptors["assay"]))
if actual_groups != expected_groups:
    raise ValueError(f"Unexpected assay/model groups: {actual_groups ^ expected_groups}")

# Map (model, assay) to a set of descriptor identities.
# Each group must have exactly 10 distinct descriptors and ranks 1 through 10.
top_sets = {}
for model in MODELS:
    for assay in ASSAYS:
        subset = top_descriptors.loc[
            (top_descriptors["model"] == model) & (top_descriptors["assay"] == assay)
        ]
        descriptors = subset["descriptor"].astype(str).tolist()
        if len(descriptors) != TOP_N or len(set(descriptors)) != TOP_N:
            raise ValueError(f"Expected {TOP_N} distinct descriptors for {model}, {assay}")
        if set(subset["rank"].astype(int)) != set(range(1, TOP_N + 1)):
            raise ValueError(f"Unexpected importance ranks for {model}, {assay}")
        top_sets[(model, assay)] = set(descriptors)


def jaccard_details(set_a, set_b):
    """Return Jaccard and the actual shared/union descriptor identities."""
    shared = set_a & set_b
    union = set_a | set_b
    return {
        "n_shared": len(shared),
        "n_union": len(union),
        "jaccard": len(shared) / len(union) if union else float("nan"),
        "shared_descriptors": "; ".join(sorted(shared)),
    }

print(f"Verified {len(top_sets)} top-{TOP_N} descriptor sets (5 assays × 2 models).")

## 3.8 — Pairwise assay comparisons (primary RF, secondary LR)

Save a **5 × 5 symmetric matrix** for each model and a **10-row detailed pairwise table** showing the exact shared descriptors. Diagonal values are 1 by definition; exclude them when reporting mean/range of cross-assay Jaccard. The ten off-diagonal pairs are the independent *pairs*, not ten statistically independent measurements for hypothesis testing.

In [ ]:
jaccard_matrices = {}
jaccard_pair_tables = {}

for model in ["RF", "LR"]:  # RF is the primary interpretation.
    mat = pd.DataFrame(1.0, index=ASSAYS, columns=ASSAYS, dtype=float)
    pair_rows = []
    for assay_a, assay_b in combinations(ASSAYS, 2):
        detail = jaccard_details(top_sets[(model, assay_a)], top_sets[(model, assay_b)])
        score = detail["jaccard"]
        mat.loc[assay_a, assay_b] = score
        mat.loc[assay_b, assay_a] = score
        pair_rows.append({"model": model, "assay_a": assay_a, "assay_b": assay_b, **detail})

    pair_table = pd.DataFrame(pair_rows).sort_values(
        ["jaccard", "assay_a", "assay_b"], ascending=[False, True, True]
    ).reset_index(drop=True)
    assert mat.shape == (5, 5) and len(pair_table) == 10
    assert np.allclose(mat.to_numpy(), mat.to_numpy().T)
    assert np.allclose(np.diag(mat), 1.0)

    jaccard_matrices[model] = mat
    jaccard_pair_tables[model] = pair_table
    mat.to_csv(metrics03 / f"jaccard_similarity_{model}.csv", index_label="assay")
    pair_table.to_csv(metrics03 / f"jaccard_pairs_{model}.csv", index=False)

    print(f"\n{model} cross-assay Jaccard — 10 pairs")
    print(f"Mean: {pair_table.jaccard.mean():.3f} | "
          f"Median: {pair_table.jaccard.median():.3f} | "
          f"Range: [{pair_table.jaccard.min():.3f}, {pair_table.jaccard.max():.3f}]")
    display(mat.round(3))
    display(pair_table.loc[:, ["assay_a", "assay_b", "n_shared", "jaccard", "shared_descriptors"]])

## 3.9 — LR vs RF within each assay (supporting)

This answers whether the two model families select similar top-10 descriptor identities **for the same biological endpoint**. It does not compare importance magnitudes, since they have different units and interpretations.

In [ ]:
cross_model_rows = []
for assay in ASSAYS:
    detail = jaccard_details(top_sets[("LR", assay)], top_sets[("RF", assay)])
    cross_model_rows.append({"assay": assay, "model_a": "LR", "model_b": "RF", **detail})

cross_model_jaccard = pd.DataFrame(cross_model_rows)
cross_model_jaccard.to_csv(metrics03 / "jaccard_similarity_LR_vs_RF.csv", index=False)
print("LR–RF top-10 overlap within each assay:")
display(cross_model_jaccard.round(3))

## 3.10 — Visualize Jaccard matrices and within-assay model agreement

The two assay matrices share a fixed color scale of 0–1. In the report, **lead with RF**; LR and LR–RF agreement are supporting analyses.

In [ ]:
def plot_jaccard_matrix(matrix, model):
    fig, ax = plt.subplots(figsize=(7.2, 6.0))
    image = ax.imshow(matrix.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(np.arange(len(ASSAYS)), labels=ASSAYS, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(ASSAYS)), labels=ASSAYS)
    for i in range(len(ASSAYS)):
        for j in range(len(ASSAYS)):
            v = float(matrix.iloc[i, j])
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v >= 0.65 else "black", fontsize=10)
    ax.set_title(f"{model}: pairwise top-{TOP_N} descriptor Jaccard similarity")
    fig.colorbar(image, ax=ax, label="Jaccard similarity (0–1)", fraction=0.045)
    fig.tight_layout()
    fig.savefig(figures03 / f"jaccard_heatmap_{model}.png", dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

for model in ["RF", "LR"]:
    plot_jaccard_matrix(jaccard_matrices[model], model)

fig, ax = plt.subplots(figsize=(8.4, 4.4))
ax.bar(cross_model_jaccard["assay"], cross_model_jaccard["jaccard"])
ax.set_ylim(0, 1)
ax.set_ylabel("Jaccard similarity (LR top 10 vs RF top 10)")
ax.set_title("Within-assay agreement between LR and RF descriptors")
for i, v in enumerate(cross_model_jaccard["jaccard"]):
    ax.text(i, v + 0.025, f"{v:.2f}", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(figures03 / "jaccard_LR_vs_RF.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(fig)
print("Saved two cross-assay Jaccard heatmaps and one LR-vs-RF comparison chart.")

---

## 3.11 — Shared-descriptor analysis: identities and overall coverage

Subsections **3.12–3.13 are two presentations of the same underlying occurrence counts**, not two unrelated statistical tests:

1. **Descriptor-level view:** identify exactly which top-10 descriptors recur across 1, 2, 3, 4 or all 5 assays, using a descriptor-by-assay matrix.
2. **Dataset-level view:** summarize how many distinct descriptors appear in **exactly** or **at least** *k* assays, as counts and percentages.

Percentages use the **number of distinct descriptors in the union of the five top-10 lists for that model** as the denominator (not the total 217 RDKit descriptors, not the 50 top-10 memberships, and not the number of molecules).

RF is the main result; LR is a secondary comparison. Presence in a top-10 list is a *model-dependent rank criterion*, not proof of chemical causation.

In [ ]:
def make_descriptor_occurrence(model):
    """One row per distinct top-10 descriptor; 0/1 presence across the five assays."""
    union = sorted(set().union(*(top_sets[(model, a)] for a in ASSAYS)))
    rows = []
    for descriptor in union:
        flags = {assay: int(descriptor in top_sets[(model, assay)]) for assay in ASSAYS}
        frequency = sum(flags.values())
        rows.append({
            "model": model,
            "descriptor": descriptor,
            **flags,
            "assay_count": frequency,
            "assay_fraction": frequency / len(ASSAYS),
            "appears_in_assays": "; ".join(a for a in ASSAYS if flags[a]),
        })
    occurrence = pd.DataFrame(rows).sort_values(
        ["assay_count", "descriptor"], ascending=[False, True]
    ).reset_index(drop=True)
    # Each of the five top-10 lists contributes exactly 10 memberships.
    assert occurrence["assay_count"].sum() == len(ASSAYS) * TOP_N
    assert all(occurrence[a].sum() == TOP_N for a in ASSAYS)
    return occurrence


def make_coverage_summary(occurrence, model):
    """Exactly-k and at-least-k occurrences; percent of UNIQUE descriptors."""
    denominator = len(occurrence)
    if denominator == 0:
        raise ValueError("Cannot summarize zero unique descriptors")
    rows = []
    for k in range(1, len(ASSAYS) + 1):
        exactly_k = int((occurrence["assay_count"] == k).sum())
        at_least_k = int((occurrence["assay_count"] >= k).sum())
        rows.append({
            "model": model,
            "k_assays": k,
            "n_unique_descriptors": denominator,
            "n_exactly_k": exactly_k,
            "pct_unique_exactly_k": exactly_k / denominator * 100,
            "n_at_least_k": at_least_k,
            "pct_unique_at_least_k": at_least_k / denominator * 100,
        })
    summary = pd.DataFrame(rows)
    assert summary["n_exactly_k"].sum() == denominator
    assert summary.loc[summary["k_assays"] == 1, "n_at_least_k"].iloc[0] == denominator
    return summary

occurrence_tables = {}
coverage_tables = {}
for model in ["RF", "LR"]:
    occurrence = make_descriptor_occurrence(model)
    coverage = make_coverage_summary(occurrence, model)
    occurrence_tables[model] = occurrence
    coverage_tables[model] = coverage
    occurrence.to_csv(metrics03 / f"descriptor_occurrence_{model}.csv", index=False)
    coverage.to_csv(metrics03 / f"descriptor_overlap_summary_{model}.csv", index=False)

    print(f"\n{model}: {len(occurrence)} distinct top-{TOP_N} descriptors in the union")
    print(f"Appearing in >=2 assays: {int(coverage.loc[coverage.k_assays==2, 'n_at_least_k'].iloc[0])} "
          f"({coverage.loc[coverage.k_assays==2, 'pct_unique_at_least_k'].iloc[0]:.1f}%)")
    print(f"Appearing in all 5 assays: {int(coverage.loc[coverage.k_assays==5, 'n_at_least_k'].iloc[0])}")
    display(coverage.round(2))
    display(occurrence.loc[occurrence["assay_count"] >= 2,
                           ["descriptor", "assay_count", "appears_in_assays"]].head(20))

## 3.12 — Descriptor-by-assay heatmaps

A filled cell means that the descriptor is in that assay's top-10 list. Rows are sorted by decreasing occurrence count. This shows **which descriptors** are repeatedly important according to each model's chosen importance method.

In [ ]:
def plot_descriptor_occurrence(occurrence, model):
    labels = occurrence["descriptor"].tolist()
    matrix = occurrence[ASSAYS].to_numpy(dtype=int)
    height = max(5.5, 0.31 * len(labels) + 1.7)
    fig, ax = plt.subplots(figsize=(10.0, height))
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(np.arange(len(ASSAYS)), labels=ASSAYS, rotation=30, ha="right")
    ax.set_yticks(np.arange(len(labels)), labels=labels, fontsize=8)
    ax.set_xlabel("Tox21 assay")
    ax.set_title(f"{model}: shared top-{TOP_N} molecular descriptors across assays")
    # Light gridlines make binary membership easier to inspect.
    ax.set_xticks(np.arange(-0.5, len(ASSAYS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(labels), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)
    # Show number of assays (2–5) in which each descriptor occurs.
    right = ax.twinx()
    right.set_ylim(ax.get_ylim())
    right.set_yticks(np.arange(len(labels)), labels=occurrence["assay_count"].tolist(), fontsize=8)
    right.set_ylabel("Number of assays")
    fig.tight_layout()
    fig.savefig(figures03 / f"descriptor_assay_heatmap_{model}.png",
                dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

for model in ["RF", "LR"]:
    plot_descriptor_occurrence(occurrence_tables[model], model)
print("Saved descriptor-by-assay membership heatmaps for RF and LR.")

## 3.13 — Frequency-distribution plots and global overlap statistics

The distribution gives counts of descriptors present in **exactly** 1, 2, 3, 4 and 5 assays. The accompanying summary gives the fraction appearing in **at least** each number of assays.

Examples of defensible statements: “X of Y distinct RF top-10 descriptors (Z%) appeared in at least two assays”; “Q descriptors occurred in all five assays.” These are **descriptive proportions**, not p-values or tests of significance.

In [ ]:
for model in ["RF", "LR"]:
    summary = coverage_tables[model]
    fig, ax = plt.subplots(figsize=(7.6, 4.8))
    bars = ax.bar(summary["k_assays"], summary["n_exactly_k"])
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_ylim(0, max(1, summary["n_exactly_k"].max()) * 1.20)
    ax.set_xlabel("Number of assays containing descriptor in their top 10")
    ax.set_ylabel("Number of unique descriptors")
    ax.set_title(f"{model}: distribution of top-10 descriptor sharing")
    for bar, count in zip(bars, summary["n_exactly_k"]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(1, summary["n_exactly_k"].max()) * 0.025,
                str(int(count)), ha="center", va="bottom")
    fig.tight_layout()
    fig.savefig(figures03 / f"descriptor_frequency_{model}.png",
                dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

# An easy-to-cite table for the Results and Discussion text.
summary_rows = []
for model in ["RF", "LR"]:
    coverage = coverage_tables[model].set_index("k_assays")
    pairs = jaccard_pair_tables[model]
    summary_rows.append({
        "model": model,
        "n_unique_top10_descriptors": int(coverage.loc[1, "n_unique_descriptors"]),
        "n_shared_at_least_2_assays": int(coverage.loc[2, "n_at_least_k"]),
        "pct_shared_at_least_2_assays": float(coverage.loc[2, "pct_unique_at_least_k"]),
        "n_shared_at_least_3_assays": int(coverage.loc[3, "n_at_least_k"]),
        "pct_shared_at_least_3_assays": float(coverage.loc[3, "pct_unique_at_least_k"]),
        "n_shared_all_5_assays": int(coverage.loc[5, "n_at_least_k"]),
        "five_way_intersection_over_union": float(coverage.loc[5, "n_at_least_k"] / coverage.loc[1, "n_unique_descriptors"]),
        "mean_pairwise_jaccard": float(pairs["jaccard"].mean()),
        "median_pairwise_jaccard": float(pairs["jaccard"].median()),
        "min_pairwise_jaccard": float(pairs["jaccard"].min()),
        "max_pairwise_jaccard": float(pairs["jaccard"].max()),
    })
overlap_research_summary = pd.DataFrame(summary_rows)
overlap_research_summary.to_csv(metrics03 / "overlap_research_question_summary.csv", index=False)
print("Summary table for the research question:")
display(overlap_research_summary.round(3))
print("Saved frequency plots and overlap_research_question_summary.csv")

## 3.14 — Interpretation for the final report

**Primary finding:** Describe the **RF** results first: mean/range of the 10 pairwise Jaccard values, the pairs with the highest/lowest overlap, the specific descriptors found in ≥2 assays, and their occurrence frequencies. Include the RF heatmap and a global overlap percentage in the main report.

**Secondary finding:** Show how LR's standardized-coefficient top-10 sets agree or differ from RF. Use the LR cross-assay Jaccard results and the within-assay LR-versus-RF comparison as supporting evidence.

**Important methodological limitations:**
- Jaccard comparisons use **exact descriptor identities**. Two correlated but different descriptors count as different, even when they encode related chemical information.
- RF permutation importance can be unstable if top-ranked features are correlated, or if the top-10 boundary contains near ties. Note particularly small, zero, or negative importance values.
- The threshold of 10 descriptors is an a priori reporting convention, not a statistical cutoff.
- These descriptive overlap summaries do **not** establish statistical significance or a common causal pathway across assays.
- Keep the held-out test set out of any subsequent model retuning; permutation importance on it is post-hoc interpretation only.

**Main new files:**

| Group | CSV outputs | Figure outputs |
| --- | --- | --- |
| Pairwise assays | `jaccard_similarity_RF.csv`, `jaccard_similarity_LR.csv`, plus `jaccard_pairs_*.csv` | `jaccard_heatmap_RF.png`, `jaccard_heatmap_LR.png` |
| Within-assay models | `jaccard_similarity_LR_vs_RF.csv` | `jaccard_LR_vs_RF.png` |
| Descriptor occurrence | `descriptor_occurrence_RF.csv`, `descriptor_occurrence_LR.csv` | `descriptor_assay_heatmap_RF.png`, `descriptor_assay_heatmap_LR.png` |
| Overall sharing | `descriptor_overlap_summary_RF.csv`, `descriptor_overlap_summary_LR.csv`, `overlap_research_question_summary.csv` | `descriptor_frequency_RF.png`, `descriptor_frequency_LR.png` |

All metrics/figures are written to the `Results_Interpretation` subfolders. No new `.py` helper files are required.